# Figure S2. SuSiE-rRSS before and after kriging correction

Supplement to Fig 1B. Uses the simulations from `02.Fig1B.Allele_Flip_Simulation` (`2026_output/Allele_flip_Gtex_Simulation`, GTEx, in-sample LD, flip rate 0.1% / 1% / 10%), and the kriging correction from `02` `[krigging_fix]`. That step fits SuSiE-RSS; this one fits SuSiE-rRSS (`R_mismatch = "eb_mix"`, `R_finite = 10000` as for in-sample LD in `09`) to:

- `flipped_rrss`: flipped $z$-scores before correction
- `kriging_rrss`: kriging-corrected $z$-scores
- `original_rrss`: original $z$-scores

Use `susie_bias_summary` from `10`. It also writes a `<method>_SER` row for each fit's SER fallback, so SuSiE-rRSS with the fallback is built from `R_reliability_flag` as in Fig S3E.

Output: `/sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/Fine_Flip_rRSS/`

## Fine-mapping

In [ ]:
[krigging_fix_rrss]
# Same inputs and kriging fix as [krigging_fix] in 02.Fig1B.Allele_Flip_Simulation, fitted with
# SuSiE-rRSS (R_mismatch = "eb_mix") instead of SuSiE-RSS.
parameter: genofile = paths
# the 02 simulation .rds files (one per region)
parameter: cwd = path("output")
parameter: job_size = 30
parameter: walltime = "100h"
parameter: mem = "40G"
parameter: numThreads = 1
parameter: R_finite = 10000
# in-sample GTEx LD: large B so 1/B is negligible (B_inf in 09 [finemap])
parameter: container = ""
input: genofile, group_by = 1
output: f'{cwd:a}/{_input[0]:bn}_krigging_fix_rrss.rds'
R:  expand = '${ }', stdout = f"{_output:n}.stdout", stderr = f"{_output:n}.stderr", container = container

    library(plink2R)
    library(tidyverse)
    library(MASS)
    library(crayon)
    library(pecotmr)
    library(susieR)
    library(vroom)
    library(pgenlibr)
    library(Rfast)

    # pecotmr GitHub version (conda not yet updated)
    for (f in list.files("/sc/arion/projects/ad-omics/haochen/rss_qc/pecotmr/R/",
                         full.names = TRUE, pattern = "\\.R$")) source(f)

    # pecotmr GitHub version (conda not yet updated)
    for (f in list.files("/sc/arion/projects/ad-omics/haochen/rss_qc/susieR/R/",
                         full.names = TRUE, pattern = "\\.R$")) source(f)
    select <- dplyr::select

    # Project utilities
    source("/sc/arion/projects/ad-omics/haochen/rss_qc/codes/code/simulate_linreg.R")
    source("/sc/arion/projects/ad-omics/haochen/rss_qc/codes/code/misc.R")
    source("/sc/arion/projects/ad-omics/haochen/rss_qc/codes/code/finemap_qc_ver2.r")
    source_build <- function(dir) {
      env <- new.env(parent = globalenv())
      for (f in list.files(dir, pattern = "\\.R$", full.names = TRUE)) sys.source(f, envir = env)
      env
    }
    susie_hc <- source_build("/sc/arion/projects/ad-omics/haochen/rss_qc/susieR/R/")$susie_rss
    path <- ${_input:r}

    # Safe wrapper: takes the susie fn explicitly, catches errors, returns NULL
    safe_susie <- function(.fn, .label, ...) {
      tryCatch(
        withCallingHandlers(
          .fn(...),
          warning = function(w) invokeRestart("muffleWarning")
        ),
        error = function(e) {
          cat(sprintf("[FAIL] %s: %s\n", .label, conditionMessage(e)))
          NULL
        }
      )
    }

    finemap_flip_susie_rrss <- function(path, B) {
      dat       <- readRDS(path)
      n         <- dat$n_samples
      true_vars <- dat$true_vars
      LD        <- dat$gtex_input$LD

      z_flipped  <- dat$gtex_input$sumstat$z
      z_original <- dat$gtex_input$sumstat_original$z

      # ----- Kriging-based sign correction on the flipped z (as in 02 [krigging_fix]) -----
      krig <- kriging_rss(z = z_flipped, R = LD, n = n)
      krig_outlier <- krig$conditional_dist %>%
        mutate(index = row_number()) %>%
        filter(abs(z) > 2 & logLR > 2)

      z_kriging <- z_flipped
      z_kriging[krig_outlier$index] <- -z_kriging[krig_outlier$index]

      # ----- SuSiE-rRSS for each z version -----
      result_list <- list(
        flipped_rrss    = safe_susie(susie_hc, "flipped_rrss",  z = z_flipped,  R = LD, n = n,
                                     R_finite = B, R_mismatch = "eb_mix"),
        original_rrss   = safe_susie(susie_hc, "original_rrss", z = z_original, R = LD, n = n,
                                     R_finite = B, R_mismatch = "eb_mix"),
        kriging_rrss    = safe_susie(susie_hc, "kriging_rrss",  z = z_kriging,  R = LD, n = n,
                                     R_finite = B, R_mismatch = "eb_mix"),
        kriging_outlier = krig_outlier
      )

      list(
        result_list = result_list,
        n_sample    = n,
        input_path  = path,
        true_vars   = true_vars
      )
    }

    finemap_result <- finemap_flip_susie_rrss(path, B = ${R_finite})

    saveRDS(finemap_result, ${_output:r})

In [ ]:
work_dir="/sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/Fine_Flip_rRSS/"
mkdir -p ${work_dir}
mkdir -p ${work_dir}/code
mkdir -p ${work_dir}/log
cd ${work_dir}/code

# Create the base_script file and write the bash code into it
cat << 'EOF' > base_script
#!/bin/bash -l
#BSUB -P acc_ad-omics
#BSUB -q premium
#BSUB -J Flip_fine_rRSS
#BSUB -n 3
#BSUB -R span[hosts=1]
#BSUB -R rusage[mem=30000]
#BSUB -W 42:00
#BSUB -o /WORK_DIR/log/Flip_fine_rRSS.%J.out
#BSUB -e /WORK_DIR/log/Flip_fine_rRSS.%J.err

export PATH=/sc/arion/projects/ad-omics/haochen/env/R_sc/bin:$PATH:/usr/bin:/bin


sos run "/sc/arion/projects/ad-omics/haochen/rss_qc/susie-rRSS-paper/simulation_code/2026_Simulation_Sinai/25.FigS2.Allele_Flip_SuSiE_rRSS.ipynb"  krigging_fix_rrss \
    --genofile `ls /sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/Allele_flip_Gtex_Simulation/result/h2g_H2G/causal_CAUSAL/miss_prop_MISSPROP_miss_fac_MISSFAC/*.rds` \
    --mem 40G \
    --cwd WORK_DIR/result/h2g_H2G/causal_CAUSAL/miss_prop_MISSPROP_miss_fac_MISSFAC/
EOF


miss_fac=-1.0

base_sh="base_script"
for miss_prop in 0.001 0.01 0.1; do
    for causal in 1 2 3; do
       for h2g in 0.05 0.1 0.15; do
            output_script="miss_prop_${miss_prop}_miss_fac_${miss_fac}_causal_${causal}_h2g_${h2g}.sh"
            cat ${base_sh}| sed "s|MISSPROP|${miss_prop}|g" |sed "s|MISSFAC|${miss_fac}|g" |sed "s|CAUSAL|${causal}|g" |sed "s|WORK_DIR|${work_dir}|g" |sed "s|H2G|${h2g}|g"   > ${output_script}
            sb ${output_script}
        done
    done
done

## Summary (`susie_bias_summary` from `10`)

Writes `Fine_Flip_rRSS/summary/result_susie_bias_summary_summary.rds`, the same layout as `Fine_Flip/summary/` read by `Data_Processing/Figure_1/Fig1B.Allele_Flip_Kriging`.

In [ ]:
work_dir="/sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/Fine_Flip_rRSS/"
mkdir -p ${work_dir}/summary
cd ${work_dir}/summary

# Create the base_script file and write the bash code into it
cat << 'EOF' > base_script
#!/bin/bash -l
#BSUB -P acc_ad-omics
#BSUB -q premium
#BSUB -J krig_rrss_summary
#BSUB -n 2
#BSUB -R span[hosts=1]
#BSUB -R rusage[mem=30000]
#BSUB -W 16:00
#BSUB -o WORK_DIR/summary/summary.%J.out
#BSUB -e WORK_DIR/summary/summary.%J.err


export PATH=/sc/arion/projects/ad-omics/haochen/env/R_sc/bin:$PATH:/usr/bin:/bin

sos run "/sc/arion/projects/ad-omics/haochen/rss_qc/susie-rRSS-paper/simulation_code/2026_Simulation_Sinai/10.Fig1B_3_S2_S5_S6.Finemapping_Summary.ipynb" susie_bias_summary \
    --folder WORK_DIR/result/ \
    --cwd WORK_DIR/summary


EOF

base_sh="base_script"

output_script="summary_susie.sh"
cat ${base_sh} |sed "s|WORK_DIR|${work_dir}|g"    > ${output_script}
sb ${output_script}